<div style="background: linear-gradient(135deg, #0f172a, #2563eb); color: white; padding: 24px 28px; border-radius: 18px; box-shadow: 0 8px 24px rgba(15, 23, 42, 0.12);">
  <h1 style="margin: 0; font-size: 30px;">02 · Data Cleaning & Integration</h1>
  <p style='margin:10px 0 0 0; opacity:0.96; line-height:1.6;'>OULAD preprocessing notebook for the final project <b>Learning Behavior Analytics and Personalized Learning Path Recommendation for At-Risk Learners</b>.</p>
  <ul style='margin:12px 0 0 18px; padding:0; line-height:1.6;'><li style='margin:6px 0;'>Cleans the 7 core OULAD tables used in the project scope.</li><li style='margin:6px 0;'>Builds intermediate analytical datasets for EDA, feature engineering, and Power BI.</li><li style='margin:6px 0;'>Keeps the workflow aligned with the project story: <b>Data Preparation → Learning Analytics → Modeling → Dashboard Storytelling</b>.</li></ul>
</div>

<div style="border-left: 5px solid #0f766e; background: #ecfeff; padding: 12px 16px; border-radius: 10px;">
  <b>Notebook design note</b><br>
  <span style="line-height:1.65;">The final project is organized around the learner–module–presentation perspective. This notebook therefore exports both cleaned source tables and convenient intermediate tables, while the EDA notebook rebuilds the canonical enrollment-level analytical base before downstream analysis.</span>
</div>

<div style="border-left: 6px solid #2563eb; background: #eff6ff; padding: 14px 18px; border-radius: 12px;">
  <h2 style="margin: 0 0 4px 0;">1. Environment setup and raw data loading</h2>
  <p style='margin:8px 0 0 0; line-height:1.6;'><b>Purpose.</b> Initialize the notebook, define data paths, load the seven raw tables, and verify that the project inputs are available before any cleaning rule is applied.</p>
  <ul style='margin:10px 0 0 18px; padding:0; line-height:1.6;'><li style='margin:4px 0;'>Tables loaded: courses, studentInfo, studentRegistration, studentVle, assessments, studentAssessment, and vle.</li><li style='margin:4px 0;'>The printed shapes provide a quick baseline for later reconciliation after cleaning.</li></ul>
</div>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style("whitegrid")

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')

ROOT = find_repo_root()
raw_data_path = ROOT / 'data' / 'raw'
cleaned_data_path = ROOT / 'data' / 'processed'
cleaned_data_path.mkdir(exist_ok=True)

In [2]:
df_courses = pd.read_csv(raw_data_path / 'courses.csv')
df_student_info = pd.read_csv(raw_data_path / 'studentInfo.csv')
df_student_registration = pd.read_csv(raw_data_path / 'studentRegistration.csv')

student_vle_source = raw_data_path / 'studentVle.csv'
student_vle_clean_fallback = cleaned_data_path / 'student_vle_clean.csv'
student_vle_summary_fallback = cleaned_data_path / 'student_vle_summary.csv'

if student_vle_source.exists():
    df_student_vle = pd.read_csv(student_vle_source)
    student_vle_source_label = 'raw/studentVle.csv'
elif student_vle_clean_fallback.exists():
    df_student_vle = pd.read_csv(student_vle_clean_fallback)
    student_vle_source_label = 'processed/student_vle_clean.csv (fallback)'
elif student_vle_summary_fallback.exists():
    df_student_vle = pd.read_csv(student_vle_summary_fallback)
    student_vle_source_label = 'processed/student_vle_summary.csv (summary fallback)'
else:
    raise FileNotFoundError('Neither raw/studentVle.csv nor processed VLE fallback files are available.')

student_vle_has_raw_events = {'sum_click', 'date'}.issubset(df_student_vle.columns)

df_assessments = pd.read_csv(raw_data_path / 'assessments.csv')
df_student_assessment = pd.read_csv(raw_data_path / 'studentAssessment.csv')
df_vle = pd.read_csv(raw_data_path / 'vle.csv')

print(f"✓ Core datasets loaded successfully!")
print(f"\nDatasets info:")
print(f"  - courses: {df_courses.shape[0]} rows × {df_courses.shape[1]} columns")
print(f"  - studentInfo: {df_student_info.shape[0]} rows × {df_student_info.shape[1]} columns")
print(f"  - studentRegistration: {df_student_registration.shape[0]} rows × {df_student_registration.shape[1]} columns")
print(f"  - studentVle: {df_student_vle.shape[0]} rows × {df_student_vle.shape[1]} columns [{student_vle_source_label}]")
print(f"  - assessments: {df_assessments.shape[0]} rows × {df_assessments.shape[1]} columns")
print(f"  - studentAssessment: {df_student_assessment.shape[0]} rows × {df_student_assessment.shape[1]} columns")
print(f"  - vle: {df_vle.shape[0]} rows × {df_vle.shape[1]} columns")

✓ Core datasets loaded successfully!

Datasets info:
  - courses: 22 rows × 3 columns
  - studentInfo: 32593 rows × 12 columns
  - studentRegistration: 32593 rows × 5 columns
  - studentVle: 10655280 rows × 6 columns [raw/studentVle.csv]
  - assessments: 206 rows × 6 columns
  - studentAssessment: 173912 rows × 5 columns
  - vle: 6364 rows × 6 columns


<div style="border-left: 6px solid #2563eb; background: #eff6ff; padding: 14px 18px; border-radius: 12px;">
  <h2 style="margin: 0 0 4px 0;">2. Raw data quality audit</h2>
  <p style='margin:8px 0 0 0; line-height:1.6;'><b>Purpose.</b> Profile duplicates and missingness at the table level so that cleaning decisions are explicit rather than ad hoc.</p>
  <ul style='margin:10px 0 0 18px; padding:0; line-height:1.6;'><li style='margin:4px 0;'>This section gives a first-pass data health scan for every dataset.</li><li style='margin:4px 0;'>Use the output here as a checkpoint before interpreting any downstream analysis.</li></ul>
</div>

In [3]:
def assess_data_quality(df, name):
    """Assess data quality of a dataset"""
    print(f"\n{name}:")
    print(f"  Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"  Duplicates: {df.duplicated().sum()}")
    print(f"  Missing values: {df.isnull().sum().sum()}")
    null_pct = (df.isnull().sum() / len(df) * 100)
    if (null_pct > 0).any():
        print(f"  Column-wise missing %:")
        print(null_pct[null_pct > 0])

assess_data_quality(df_student_info, "Student Info")
assess_data_quality(df_student_registration, "Student Registration")
assess_data_quality(df_student_vle, "Student VLE")
assess_data_quality(df_assessments, "Assessments")
assess_data_quality(df_student_assessment, "Student Assessment")
assess_data_quality(df_vle, "VLE")
assess_data_quality(df_courses, "Courses")


Student Info:
  Shape: 32593 rows × 12 columns
  Duplicates: 0
  Missing values: 1111
  Column-wise missing %:
imd_band    3.408707
dtype: float64

Student Registration:
  Shape: 32593 rows × 5 columns
  Duplicates: 0
  Missing values: 22566
  Column-wise missing %:
date_registration       0.138066
date_unregistration    69.097659
dtype: float64

Student VLE:
  Shape: 10655280 rows × 6 columns


  Duplicates: 787170
  Missing values: 0



Assessments:
  Shape: 206 rows × 6 columns
  Duplicates: 0
  Missing values: 11
  Column-wise missing %:
date    5.339806
dtype: float64

Student Assessment:
  Shape: 173912 rows × 5 columns
  Duplicates: 0
  Missing values: 173
  Column-wise missing %:
score    0.099476
dtype: float64

VLE:
  Shape: 6364 rows × 6 columns
  Duplicates: 0
  Missing values: 10486
  Column-wise missing %:
week_from    82.385292
week_to      82.385292
dtype: float64

Courses:
  Shape: 22 rows × 3 columns
  Duplicates: 0
  Missing values: 0


<div style="border-left: 6px solid #0f766e; background: #ecfdf5; padding: 14px 18px; border-radius: 12px;">
  <h2 style="margin: 0 0 4px 0;">3. Cleaning registration records</h2>
  <p style='margin:8px 0 0 0; line-height:1.6;'><b>Purpose.</b> Prepare student registration data because it anchors course participation, registration timing, and withdrawal behavior.</p>
  <ul style='margin:10px 0 0 18px; padding:0; line-height:1.6;'><li style='margin:4px 0;'>Remove duplicate registration records at the working key used in the current notebook.</li><li style='margin:4px 0;'>Inspect missing registration dates and invalid registration–unregistration order.</li><li style='margin:4px 0;'>Preserve timing variables needed later for persistence and at-risk analysis.</li></ul>
</div>

In [4]:
df_student_registration_clean = df_student_registration.copy()

initial_rows = len(df_student_registration_clean)
df_student_registration_clean = df_student_registration_clean.drop_duplicates(
    subset=['id_student', 'code_module', 'code_presentation'], keep='first')
print(f"✓ Removed {initial_rows - len(df_student_registration_clean)} duplicate registrations")

print(f"\nMissing values:")
print(df_student_registration_clean[['date_registration', 'date_unregistration']].isnull().sum())

for module in df_student_registration_clean['code_module'].unique():
    for presentation in df_student_registration_clean['code_presentation'].unique():
        mask = (df_student_registration_clean['code_module'] == module) & \
               (df_student_registration_clean['code_presentation'] == presentation)
        median_date = df_student_registration_clean.loc[mask, 'date_registration'].median()
        df_student_registration_clean.loc[mask & df_student_registration_clean['date_registration'].isnull(), 'date_registration'] = median_date

print(f"\n✓ Student Registration cleaned: {len(df_student_registration_clean)} records")
print(f"Missing date_registration values filled with module-presentation median")

invalid_dates = (df_student_registration_clean['date_unregistration'] < df_student_registration_clean['date_registration']).sum()
print(f"Invalid date relationships (unregistration < registration): {invalid_dates}")

if invalid_dates > 0:
    print("Fixing invalid date relationships...")
    df_student_registration_clean.loc[
        df_student_registration_clean['date_unregistration'] < df_student_registration_clean['date_registration'],
        'date_unregistration'
    ] = np.nan

✓ Removed 0 duplicate registrations

Missing values:
date_registration         45
date_unregistration    22521
dtype: int64

✓ Student Registration cleaned: 32593 records
Missing date_registration values filled with module-presentation median
Invalid date relationships (unregistration < registration): 21
Fixing invalid date relationships...


<div style="border-left: 6px solid #0f766e; background: #ecfdf5; padding: 14px 18px; border-radius: 12px;">
  <h2 style="margin: 0 0 4px 0;">4. Cleaning VLE catalog metadata</h2>
  <p style='margin:8px 0 0 0; line-height:1.6;'><b>Purpose.</b> Standardize the VLE site catalog so content types and learning-resource time windows can be used safely in EDA and recommendation design.</p>
  <ul style='margin:10px 0 0 18px; padding:0; line-height:1.6;'><li style='margin:4px 0;'>Resolve duplicates at the site level.</li><li style='margin:4px 0;'>Repair or normalize week ranges and convert activity types into categorical variables.</li></ul>
</div>

In [5]:
df_vle_clean = df_vle.copy()

initial_rows = len(df_vle_clean)
df_vle_clean = df_vle_clean.drop_duplicates(
    subset=['id_site', 'code_module', 'code_presentation'], keep='first')
print(f"✓ Removed {initial_rows - len(df_vle_clean)} duplicate VLE site records")

print(f"\nMissing values:")
print(df_vle_clean[['week_from', 'week_to']].isnull().sum())

df_vle_clean['week_from'] = df_vle_clean['week_from'].fillna(0).astype(int)
df_vle_clean['week_to'] = df_vle_clean['week_to'].fillna(0).astype(int)

invalid_weeks = (df_vle_clean['week_from'] > df_vle_clean['week_to']).sum()
print(f"Invalid week relationships (week_from > week_to): {invalid_weeks}")

if invalid_weeks > 0:
    print("Fixing invalid week relationships...")
    mask = df_vle_clean['week_from'] > df_vle_clean['week_to']
    df_vle_clean.loc[mask, ['week_from', 'week_to']] = df_vle_clean.loc[mask, ['week_to', 'week_from']].values

df_vle_clean['activity_type'] = df_vle_clean['activity_type'].astype('category')

print(f"\n✓ VLE cleaned: {len(df_vle_clean)} records")
print(f"Activity types: {df_vle_clean['activity_type'].nunique()}")
print(f"Week range: {df_vle_clean['week_from'].min()} - {df_vle_clean['week_to'].max()}")

✓ Removed 0 duplicate VLE site records

Missing values:
week_from    5243
week_to      5243
dtype: int64
Invalid week relationships (week_from > week_to): 0

✓ VLE cleaned: 6364 records
Activity types: 20
Week range: 0 - 29


<div style="border-left: 6px solid #0f766e; background: #ecfdf5; padding: 14px 18px; border-radius: 12px;">
  <h2 style="margin: 0 0 4px 0;">5. Cleaning clickstream interaction logs</h2>
  <p style='margin:8px 0 0 0; line-height:1.6;'><b>Purpose.</b> Prepare the high-volume studentVle table for behavior analytics while preserving the signal needed for engagement, recency, and early-warning features.</p>
  <ul style='margin:10px 0 0 18px; padding:0; line-height:1.6;'><li style='margin:4px 0;'>Separate structural validation from analytical outlier review.</li><li style='margin:4px 0;'>Flag the upper tail of click volume for interpretation, but keep the clickstream usable for later aggregation.</li></ul>
</div>

In [6]:
df_student_vle_clean = df_student_vle.copy()

print(f"Initial shape: {df_student_vle_clean.shape}")

if student_vle_has_raw_events:
    print(f"\nData validation:")
    print(f"  Sum clicks - Min: {df_student_vle_clean['sum_click'].min()}, Max: {df_student_vle_clean['sum_click'].max()}")
    print(f"  Date range: {df_student_vle_clean['date'].min()} to {df_student_vle_clean['date'].max()}")

    initial_rows = len(df_student_vle_clean)
    df_student_vle_clean = df_student_vle_clean[df_student_vle_clean['sum_click'] > 0]
    print(f"\n✓ Removed {initial_rows - len(df_student_vle_clean)} records with 0 clicks")

    vle_event_key = ['code_module', 'code_presentation', 'id_student', 'id_site', 'date']
    duplicate_event_rows = df_student_vle_clean.duplicated(subset=vle_event_key, keep=False).sum()
    pre_aggregate_rows = len(df_student_vle_clean)
    df_student_vle_clean = (
        df_student_vle_clean
        .groupby(vle_event_key, as_index=False)['sum_click']
        .sum()
    )
    print(
        f"✓ Collapsed {duplicate_event_rows:,} rows that shared the same student-site-day key "
        f"into {len(df_student_vle_clean):,} cleaned event rows"
    )
    print(f"  Rows consolidated during aggregation: {pre_aggregate_rows - len(df_student_vle_clean):,}")

    Q1 = df_student_vle_clean['sum_click'].quantile(0.25)
    Q3 = df_student_vle_clean['sum_click'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_student_vle_clean[
        (df_student_vle_clean['sum_click'] < lower_bound) |
        (df_student_vle_clean['sum_click'] > upper_bound)
    ]
    print(f"\nOutlier detection (IQR method):")
    print(f"  Lower bound: {lower_bound:.0f}, Upper bound: {upper_bound:.0f}")
    print(f"  Outliers detected: {len(outliers)} records")
    print(f"  Outlier range: {outliers['sum_click'].min()} - {outliers['sum_click'].max()}")

    p99 = df_student_vle_clean['sum_click'].quantile(0.99)
    print(f"  99th percentile: {p99:.0f}")

    print(f"\n✓ Student VLE cleaned: {len(df_student_vle_clean)} records")
    print(f"Missing values: {df_student_vle_clean.isnull().sum().sum()}")
else:
    summary_cols = [
        'id_student', 'code_module', 'code_presentation',
        'total_clicks', 'avg_clicks', 'max_clicks_per_day',
        'first_activity_date', 'last_activity_date'
    ]
    missing_summary_cols = [col for col in summary_cols if col not in df_student_vle_clean.columns]
    if missing_summary_cols:
        raise ValueError(f'VLE summary fallback is missing columns: {missing_summary_cols}')

    initial_rows = len(df_student_vle_clean)
    df_student_vle_clean = df_student_vle_clean[summary_cols].drop_duplicates(
        subset=['id_student', 'code_module', 'code_presentation'], keep='first'
    )

    print("\nSummary fallback detected:")
    print(f"  Source: {student_vle_source_label}")
    print(f"  Removed {initial_rows - len(df_student_vle_clean)} duplicate summary rows")
    print(f"  Enrollment summaries available: {len(df_student_vle_clean)}")
    print(
        f"  Activity date range: {df_student_vle_clean['first_activity_date'].min()} "
        f"to {df_student_vle_clean['last_activity_date'].max()}"
    )
    print("  Raw day-level clickstream is unavailable in this workspace, so clickstream cleaning is skipped.")

Initial shape: (10655280, 6)

Data validation:
  Sum clicks - Min: 1, Max: 6977
  Date range: -25 to 269



✓ Removed 0 records with 0 clicks


✓ Collapsed 3,810,465 rows that shared the same student-site-day key into 8,459,320 cleaned event rows
  Rows consolidated during aggregation: 2,195,960

Outlier detection (IQR method):
  Lower bound: -4, Upper bound: 8
  Outliers detected: 1048460 records
  Outlier range: 9 - 6977
  99th percentile: 44

✓ Student VLE cleaned: 8459320 records


Missing values: 0


<div style="border-left: 6px solid #0f766e; background: #ecfdf5; padding: 14px 18px; border-radius: 12px;">
  <h2 style="margin: 0 0 4px 0;">6. Cleaning assessment metadata</h2>
  <p style='margin:8px 0 0 0; line-height:1.6;'><b>Purpose.</b> Validate assessment dates, types, and weights so submission performance can later be interpreted in the correct course context.</p>
  <ul style='margin:10px 0 0 18px; padding:0; line-height:1.6;'><li style='margin:4px 0;'>Check duplicates and missing assessment dates.</li><li style='margin:4px 0;'>Keep weight information consistent for score aggregation and discipline-related features.</li></ul>
</div>

In [7]:
df_assessments_clean = df_assessments.copy()

initial_rows = len(df_assessments_clean)
df_assessments_clean = df_assessments_clean.drop_duplicates(
    subset=['id_assessment', 'code_module', 'code_presentation'], keep='first')
print(f"✓ Removed {initial_rows - len(df_assessments_clean)} duplicate assessment records")

print(f"\nMissing values:")
print(df_assessments_clean.isnull().sum())

df_assessments_clean['date'] = df_assessments_clean['date'].fillna(0).astype(int)

print(f"\nWeight validation:")
print(f"  Min: {df_assessments_clean['weight'].min()}, Max: {df_assessments_clean['weight'].max()}")
print(f"  Sum by module:")
print(df_assessments_clean.groupby('code_module')['weight'].sum())

df_assessments_clean['assessment_type'] = df_assessments_clean['assessment_type'].astype('category')

print(f"\n✓ Assessments cleaned: {len(df_assessments_clean)} records")
print(f"Assessment types: {df_assessments_clean['assessment_type'].nunique()}")

✓ Removed 0 duplicate assessment records

Missing values:
code_module           0
code_presentation     0
id_assessment         0
assessment_type       0
date                 11
weight                0
dtype: int64

Weight validation:
  Min: 0.0, Max: 100.0
  Sum by module:
code_module
AAA    400.0
BBB    800.0
CCC    600.0
DDD    800.0
EEE    600.0
FFF    800.0
GGG    300.0
Name: weight, dtype: float64

✓ Assessments cleaned: 206 records
Assessment types: 3


<div style="border-left: 6px solid #0f766e; background: #ecfdf5; padding: 14px 18px; border-radius: 12px;">
  <h2 style="margin: 0 0 4px 0;">7. Cleaning student assessment submissions</h2>
  <p style='margin:8px 0 0 0; line-height:1.6;'><b>Purpose.</b> Prepare learner submission records for score analysis, lateness logic, and performance-driven risk exploration.</p>
  <ul style='margin:10px 0 0 18px; padding:0; line-height:1.6;'><li style='margin:4px 0;'>Remove duplicate submissions under the notebook's current rule.</li><li style='margin:4px 0;'>Check missing or invalid scores and keep the banking flag ready for later use.</li></ul>
</div>

In [8]:
df_student_assessment_clean = df_student_assessment.copy()

print(f"Initial shape: {df_student_assessment_clean.shape}")

initial_rows = len(df_student_assessment_clean)
df_student_assessment_clean = df_student_assessment_clean.drop_duplicates(
    subset=['id_student', 'id_assessment', 'date_submitted'], keep='first')
print(f"✓ Removed {initial_rows - len(df_student_assessment_clean)} duplicate submissions")

print(f"\nMissing values:")
print(df_student_assessment_clean.isnull().sum())

missing_scores = df_student_assessment_clean['score'].isnull().sum()
print(f"\nMissing score values: {missing_scores}")

df_student_assessment_clean['score'] = df_student_assessment_clean['score'].fillna(0)

print(f"\nScore validation:")
print(f"  Min: {df_student_assessment_clean['score'].min()}, Max: {df_student_assessment_clean['score'].max()}")

invalid_scores = df_student_assessment_clean[
    (df_student_assessment_clean['score'] < 0) | 
    (df_student_assessment_clean['score'] > 100)
]
print(f"  Invalid scores (out of 0-100): {len(invalid_scores)}")

if len(invalid_scores) > 0:
    print(f"  Score range: {invalid_scores['score'].min()} - {invalid_scores['score'].max()}")

df_student_assessment_clean['is_banked'] = df_student_assessment_clean['is_banked'].astype('int8')

print(f"\n✓ Student Assessment cleaned: {len(df_student_assessment_clean)} records")
print(f"Score distribution:")
print(df_student_assessment_clean['score'].describe())

Initial shape: (173912, 5)
✓ Removed 0 duplicate submissions

Missing values:
id_assessment       0
id_student          0
date_submitted      0
is_banked           0
score             173
dtype: int64

Missing score values: 173

Score validation:
  Min: 0.0, Max: 100.0
  Invalid scores (out of 0-100): 0

✓ Student Assessment cleaned: 173912 records
Score distribution:
count    173912.000000
mean         75.724171
std          18.940093
min           0.000000
25%          65.000000
50%          80.000000
75%          90.000000
max         100.000000
Name: score, dtype: float64


<div style="border-left: 6px solid #0f766e; background: #ecfdf5; padding: 14px 18px; border-radius: 12px;">
  <h2 style="margin: 0 0 4px 0;">8. Cleaning learner profile / outcome data</h2>
  <p style='margin:8px 0 0 0; line-height:1.6;'><b>Purpose.</b> Standardize the learner-facing background fields and final outcome labels used across EDA, segmentation, and at-risk analysis.</p>
  <ul style='margin:10px 0 0 18px; padding:0; line-height:1.6;'><li style='margin:4px 0;'>Fill missing categorical background fields with an explicit placeholder.</li><li style='margin:4px 0;'>Convert repeated text fields into categorical columns for cleaner summaries and lighter memory use.</li></ul>
</div>

In [9]:
df_student_info_clean = df_student_info.copy()

enrollment_key = ['code_module', 'code_presentation', 'id_student']

duplicate_enrollment_rows = df_student_info_clean.duplicated(subset=enrollment_key).sum()
df_student_info_clean = df_student_info_clean.drop_duplicates(
    subset=enrollment_key,
    keep='first'
)

info_fill_cols = [
    'gender', 'region', 'highest_education', 'imd_band',
    'age_band', 'disability', 'final_result'
]
df_student_info_clean[info_fill_cols] = df_student_info_clean[info_fill_cols].fillna('Unknown')

for col in info_fill_cols:
    df_student_info_clean[col] = df_student_info_clean[col].astype('category')

print(f"✓ student_info_clean ready: {len(df_student_info_clean)} enrollment rows")
print(f"  Duplicate enrollment rows removed: {duplicate_enrollment_rows}")
print(f"  Unique learners retained: {df_student_info_clean['id_student'].nunique()}")
print(df_student_info_clean.isnull().sum())

✓ student_info_clean ready: 32593 enrollment rows
  Duplicate enrollment rows removed: 0
  Unique learners retained: 28785
code_module             0
code_presentation       0
id_student              0
gender                  0
region                  0
highest_education       0
imd_band                0
age_band                0
num_of_prev_attempts    0
studied_credits         0
disability              0
final_result            0
dtype: int64


<div style="border-left: 5px solid #b45309; background: #fffbeb; padding: 12px 16px; border-radius: 10px;">
  <b>Important downstream reminder</b><br>
  <span style="line-height:1.65;">For project interpretation, keep in mind that OULAD analyses are strongest when behavior and outcomes are read at the enrollment grain. The merged EDA notebook explicitly reconstructs that grain before producing the final insight layer.</span>
</div>

<div style="border-left: 6px solid #7c3aed; background: #f5f3ff; padding: 14px 18px; border-radius: 12px;">
  <h2 style="margin: 0 0 4px 0;">9. Build integrated analytical datasets</h2>
  <p style='margin:8px 0 0 0; line-height:1.6;'><b>Purpose.</b> Create reusable outputs that bridge raw cleaned tables and the next analytical notebooks.</p>
  <ul style='margin:10px 0 0 18px; padding:0; line-height:1.6;'><li style='margin:4px 0;'>Student profile view for high-level learner descriptors.</li><li style='margin:4px 0;'>Student–course registration and assessment-performance tables for reporting and modeling.</li><li style='margin:4px 0;'>A compact VLE summary table for fast engagement analysis.</li></ul>
</div>

In [10]:
enrollment_key = ['code_module', 'code_presentation', 'id_student']

def mode_or_first(series):
    non_null = series.dropna()
    if non_null.empty:
        return np.nan
    mode = non_null.mode()
    return mode.iloc[0] if not mode.empty else non_null.iloc[0]

student_profile_base = (
    df_student_info_clean.groupby('id_student', as_index=False)
    .agg({
        'gender': mode_or_first,
        'region': mode_or_first,
        'highest_education': mode_or_first,
        'imd_band': mode_or_first,
        'age_band': mode_or_first,
        'disability': mode_or_first,
    })
)

registration_summary = (
    df_student_info_clean.groupby('id_student', as_index=False)
    .agg(
        num_modules=('code_module', 'nunique'),
        num_presentations=('code_presentation', 'nunique'),
        num_enrollments=('code_module', 'size'),
        max_prev_attempts=('num_of_prev_attempts', 'max'),
        max_studied_credits=('studied_credits', 'max'),
    )
)

df_student_profile = student_profile_base.merge(registration_summary, on='id_student', how='left')

print(f"  Student Profile: {len(df_student_profile)} learners")
print(f"  Columns: {df_student_profile.shape[1]}")

print("\n✓ Creating Student-Course Registration Dataset...")

course_join_cols = enrollment_key + [
    'gender', 'region', 'highest_education', 'imd_band', 'age_band',
    'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result'
]
df_student_course_reg = df_student_registration_clean.merge(
    df_student_info_clean[course_join_cols],
    on=enrollment_key,
    how='left'
)

print(f"  Student-Course Registration: {len(df_student_course_reg)} records")
print(f"  Missing matched outcomes: {df_student_course_reg['final_result'].isna().sum()}")

print("\n✓ Creating Assessment Performance Dataset...")

df_assessment_perf = (
    df_student_assessment_clean
    .merge(
        df_assessments_clean[['id_assessment', 'code_module', 'code_presentation', 'assessment_type', 'weight']],
        on='id_assessment',
        how='left'
    )
    .merge(
        df_student_info_clean[enrollment_key + ['final_result']],
        on=enrollment_key,
        how='left'
    )
)

print(f"  Assessment Performance: {len(df_assessment_perf)} records")
print(f"  Missing matched outcomes: {df_assessment_perf['final_result'].isna().sum()}")

print("\n✓ Creating Student-VLE Activity Dataset...")

if student_vle_has_raw_events:
    df_student_vle_summary = (
        df_student_vle_clean.groupby(enrollment_key)
        .agg({
            'sum_click': ['sum', 'mean', 'max'],
            'date': ['min', 'max']
        })
        .reset_index()
    )

    df_student_vle_summary.columns = [
        'code_module', 'code_presentation', 'id_student',
        'total_clicks', 'avg_clicks', 'max_clicks_per_day',
        'first_activity_date', 'last_activity_date'
    ]
    df_student_vle_summary = df_student_vle_summary[
        ['id_student', 'code_module', 'code_presentation', 'total_clicks', 'avg_clicks', 'max_clicks_per_day', 'first_activity_date', 'last_activity_date']
    ]
else:
    df_student_vle_summary = df_student_vle_clean[
        ['id_student', 'code_module', 'code_presentation', 'total_clicks', 'avg_clicks', 'max_clicks_per_day', 'first_activity_date', 'last_activity_date']
    ].copy()
    print("  Using pre-aggregated VLE summary fallback (raw clickstream not available).")

print(f"  Student VLE Summary: {len(df_student_vle_summary)} records")

print(f"\n✓ Data Integration Complete!")
print(f"  - Student Profile: {len(df_student_profile)} learners")
print(f"  - Student-Course Registrations: {len(df_student_course_reg)} enrollments")
print(f"  - Assessment Results: {len(df_assessment_perf)} submissions")
print(f"  - VLE Activity: {len(df_student_vle_summary)} student-course VLE summaries")

  Student Profile: 28785 learners
  Columns: 12

✓ Creating Student-Course Registration Dataset...
  Student-Course Registration: 32593 records
  Missing matched outcomes: 0

✓ Creating Assessment Performance Dataset...
  Assessment Performance: 173912 records
  Missing matched outcomes: 0

✓ Creating Student-VLE Activity Dataset...


  Student VLE Summary: 29228 records

✓ Data Integration Complete!
  - Student Profile: 28785 learners
  - Student-Course Registrations: 32593 enrollments
  - Assessment Results: 173912 submissions
  - VLE Activity: 29228 student-course VLE summaries


<div style="border-left: 6px solid #7c3aed; background: #f5f3ff; padding: 14px 18px; border-radius: 12px;">
  <h2 style="margin: 0 0 4px 0;">10. Post-integration integrity checks</h2>
  <p style='margin:8px 0 0 0; line-height:1.6;'><b>Purpose.</b> Validate cross-table consistency after cleaning so the exported datasets are analytically coherent.</p>
  <ul style='margin:10px 0 0 18px; padding:0; line-height:1.6;'><li style='margin:4px 0;'>Check learner coverage across registration, profile, VLE, and assessment tables.</li><li style='margin:4px 0;'>Verify module coverage across the main behavioral and academic sources.</li></ul>
</div>

In [11]:
def validate_data_relationships(df_reg, df_info, df_vle, df_assess_raw, df_assess_student):
    """Validate data relationships and integrity at the enrollment grain."""
    enrollment_key = ['code_module', 'code_presentation', 'id_student']
    print("\n✓ Referential Integrity Checks:")

    reg_enrollments = set(map(tuple, df_reg[enrollment_key].drop_duplicates().to_numpy()))
    info_enrollments = set(map(tuple, df_info[enrollment_key].drop_duplicates().to_numpy()))
    missing_info = reg_enrollments - info_enrollments
    print(f"  Registration enrollments missing from info: {len(missing_info)}")

    vle_enrollments = set(map(tuple, df_vle[enrollment_key].drop_duplicates().to_numpy()))
    missing_reg = vle_enrollments - reg_enrollments
    print(f"  VLE enrollments missing from registration: {len(missing_reg)}")

    assess_enrollments = (
        df_assess_student
        .merge(df_assess_raw[['id_assessment', 'code_module', 'code_presentation']], on='id_assessment', how='left')
        [enrollment_key]
        .drop_duplicates()
    )
    assess_enrollment_set = set(map(tuple, assess_enrollments.to_numpy()))
    missing_info_assess = assess_enrollment_set - info_enrollments
    print(f"  Assessment enrollments missing from info: {len(missing_info_assess)}")

    print(f"\n✓ Module Consistency:")
    reg_modules = set(df_reg['code_module'].unique())
    assess_modules = set(df_assess_raw['code_module'].unique())
    vle_modules = set(df_vle['code_module'].unique())
    
    print(f"  Unique modules in registration: {len(reg_modules)}")
    print(f"  Unique modules in assessments: {len(assess_modules)}")
    print(f"  Unique modules in VLE: {len(vle_modules)}")
    print(f"  Module coverage: {len(reg_modules & assess_modules & vle_modules)}/{len(reg_modules)}")
    
    return missing_info, missing_reg, missing_info_assess

missing_info, missing_reg, missing_info_assess = validate_data_relationships(
    df_student_registration_clean,
    df_student_info_clean,
    df_student_vle_clean,
    df_assessments_clean,
    df_student_assessment_clean
)

print("\n✓ Integrated dataset checks:")
print(f"  Student-course registration rows without outcome match: {df_student_course_reg['final_result'].isna().sum()}")
print(f"  Assessment performance rows without outcome match: {df_assessment_perf['final_result'].isna().sum()}")

print("\n✓ Data Completeness Summary (After Cleaning):")
print(f"  Student Info: {len(df_student_info_clean)} records, {df_student_info_clean.isnull().sum().sum()} missing values")
print(f"  Student Registration: {len(df_student_registration_clean)} records, {df_student_registration_clean.isnull().sum().sum()} missing values")
print(f"  Student VLE ({student_vle_source_label}): {len(df_student_vle_clean)} records, {df_student_vle_clean.isnull().sum().sum()} missing values")
if not student_vle_has_raw_events:
    print("  Note: VLE is only available as enrollment-level summaries in this workspace.")
print(f"  Assessments: {len(df_assessments_clean)} records, {df_assessments_clean.isnull().sum().sum()} missing values")
print(f"  Student Assessment: {len(df_student_assessment_clean)} records, {df_student_assessment_clean.isnull().sum().sum()} missing values")
print(f"  VLE: {len(df_vle_clean)} records, {df_vle_clean.isnull().sum().sum()} missing values")


✓ Referential Integrity Checks:
  Registration enrollments missing from info: 0


  VLE enrollments missing from registration: 0
  Assessment enrollments missing from info: 0

✓ Module Consistency:
  Unique modules in registration: 7
  Unique modules in assessments: 7
  Unique modules in VLE: 7
  Module coverage: 7/7

✓ Integrated dataset checks:
  Student-course registration rows without outcome match: 0
  Assessment performance rows without outcome match: 0

✓ Data Completeness Summary (After Cleaning):
  Student Info: 32593 records, 0 missing values
  Student Registration: 32593 records, 22542 missing values


  Student VLE (raw/studentVle.csv): 8459320 records, 0 missing values
  Assessments: 206 records, 0 missing values
  Student Assessment: 173912 records, 0 missing values
  VLE: 6364 records, 0 missing values


<div style="border-left: 6px solid #7c3aed; background: #f5f3ff; padding: 14px 18px; border-radius: 12px;">
  <h2 style="margin: 0 0 4px 0;">11. Export processed datasets and handoff</h2>
  <p style='margin:8px 0 0 0; line-height:1.6;'><b>Purpose.</b> Save both cleaned source tables and integrated datasets for the next notebooks in the pipeline.</p>
  <ul style='margin:10px 0 0 18px; padding:0; line-height:1.6;'><li style='margin:4px 0;'>These files are the immediate inputs for the merged EDA notebook.</li><li style='margin:4px 0;'>The final printed summary also provides a quick reconciliation checkpoint before moving on.</li></ul>
</div>

In [12]:
df_student_info_clean.to_csv(cleaned_data_path / 'student_info_clean.csv', index=False)
df_student_registration_clean.to_csv(cleaned_data_path / 'student_registration_clean.csv', index=False)
if student_vle_has_raw_events:
    df_student_vle_clean.to_csv(cleaned_data_path / 'student_vle_clean.csv', index=False)
df_assessments_clean.to_csv(cleaned_data_path / 'assessments_clean.csv', index=False)
df_student_assessment_clean.to_csv(cleaned_data_path / 'student_assessment_clean.csv', index=False)
df_vle_clean.to_csv(cleaned_data_path / 'vle_clean.csv', index=False)
df_courses.to_csv(cleaned_data_path / 'courses.csv', index=False)

print("✓ Saved cleaned raw tables:")
print(f"  - student_info_clean.csv")
print(f"  - student_registration_clean.csv")
if student_vle_has_raw_events:
    print(f"  - student_vle_clean.csv")
else:
    print(f"  - student_vle_clean.csv (skipped: raw clickstream unavailable)")
print(f"  - assessments_clean.csv")
print(f"  - student_assessment_clean.csv")
print(f"  - vle_clean.csv")
print(f"  - courses.csv")

df_student_profile.to_csv(cleaned_data_path / 'student_profile.csv', index=False)
df_student_course_reg.to_csv(cleaned_data_path / 'student_course_registration.csv', index=False)
df_assessment_perf.to_csv(cleaned_data_path / 'assessment_performance.csv', index=False)
df_student_vle_summary.to_csv(cleaned_data_path / 'student_vle_summary.csv', index=False)

print("\n✓ Saved integrated datasets:")
print(f"  - student_profile.csv")
print(f"  - student_course_registration.csv")
print(f"  - assessment_performance.csv")
print(f"  - student_vle_summary.csv")
print(f"Total enrollments in student_info_clean: {len(df_student_info_clean)}")
print(f"Unique learners: {df_student_info_clean['id_student'].nunique()}")
print(f"Unique modules: {df_student_info_clean['code_module'].nunique()}")
print(f"Unique module-presentations: {df_student_info_clean[['code_module', 'code_presentation']].drop_duplicates().shape[0]}")
print(f"Total registrations: {len(df_student_registration_clean)}")
if student_vle_has_raw_events:
    print(f"Total VLE interactions: {len(df_student_vle_clean)}")
else:
    print(f"Total VLE summaries available: {len(df_student_vle_summary)}")
print(f"Total assessment submissions: {len(df_student_assessment_clean)}")
print(f"\nEnrollment-level target variable distribution:")
print(df_student_info_clean['final_result'].value_counts())

✓ Saved cleaned raw tables:
  - student_info_clean.csv
  - student_registration_clean.csv
  - student_vle_clean.csv
  - assessments_clean.csv
  - student_assessment_clean.csv
  - vle_clean.csv
  - courses.csv



✓ Saved integrated datasets:
  - student_profile.csv
  - student_course_registration.csv
  - assessment_performance.csv
  - student_vle_summary.csv
Total enrollments in student_info_clean: 32593
Unique learners: 28785
Unique modules: 7
Unique module-presentations: 22
Total registrations: 32593
Total VLE interactions: 8459320
Total assessment submissions: 173912

Enrollment-level target variable distribution:
final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64


<div style="border-left: 5px solid #1d4ed8; background: #eff6ff; padding: 12px 16px; border-radius: 10px;">
  <b>Next notebook</b><br>
  <span style="line-height:1.65;">Proceed to the merged EDA notebook to translate these processed tables into report-ready insights, feature candidates, and a dashboard story aligned with the project proposal.</span>
</div>